# HighRes-Net Training on Microscopy Data

This notebook trains HighRes-Net on your TIFF microscopy dataset (7 LR frames → 2x HR upsampling).

**Format:**
- ✓ TIFF 8-bit grayscale files (LR_1.tif/tiff through LR_7.tif/tiff + HR.tif/tiff per image)
- ✓ Pre-augmented images in D:\GUC\Datasets\HighRes input test\image_0000\, image_0001\, etc.
- ✓ 2x upsampling (128×128 LR → 256×256 HR)
- ✓ Supports both .tif and .tiff extensions (identical format)
- ✓ GPU acceleration with RTX 4060

## 1. Setup and Imports

In [1]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import shutil
import time

sys.path.insert(0, '../src')

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from DataLoader import TiffPatchDataset, collateFunction
from DeepNetworks.HRNet import HRNet

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch Version: 2.7.1+cu118
CUDA Available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB


## 2. Verify Dataset

In [2]:
data_root = Path("D:\\GUC\\Datasets\\HighRes Train input")
image_dirs = sorted([d for d in data_root.iterdir() if d.is_dir() and d.name.startswith("image_")])

print(f"✓ Found {len(image_dirs)} images")

# Verify first image has required files
if image_dirs:
    first = image_dirs[0]
    lr_files = sorted(first.glob("LR_*.tif*"))
    hr_file = list(first.glob("HR.tif*"))
    print(f"✓ Sample ({first.name}): {len(lr_files)} LR frames, {len(hr_file)} HR frame")

✓ Found 247 images
✓ Sample (image_0000): 7 LR frames, 1 HR frame


## 3. Load Configuration

In [3]:
config_path = Path('../config/config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

print("Training Configuration:")
print(f"  Num epochs: {config['training']['num_epochs']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['lr']}")
print(f"  Patch size: {config['training']['patch_size']}")
print(f"  N_views: {config['training']['n_views']}")
print(f"  Create patches: {config['training']['create_patches']}")

Training Configuration:
  Num epochs: 35
  Batch size: 8
  Learning rate: 3e-05
  Patch size: 128
  N_views: 7
  Create patches: False


## 5. Create Datasets and DataLoaders

In [4]:
print("Creating TIFF dataset...")

# Discover all image directories
data_root = Path("D:\\GUC\\Datasets\\HighRes Train input")
image_dirs = sorted([str(d) for d in data_root.iterdir() if d.is_dir() and d.name.startswith("image_")])

print(f"Discovered {len(image_dirs)} images")

# Create dataset from images
train_dataset = TiffPatchDataset(
    patch_dirs=image_dirs,
    max_views=config['training']['n_views']
)

print(f"Training dataset size: {len(train_dataset)} images")

# Create DataLoader
batch_size = config['training']['batch_size']
min_L = config['training']['min_L']

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    collate_fn=collateFunction(min_L=min_L),
    pin_memory=torch.cuda.is_available()
)

print(f"DataLoader created:")
print(f"  batch_size: {batch_size}")
print(f"  num_batches: {len(train_loader)}")
print(f"  pin_memory: {torch.cuda.is_available()}")

Creating TIFF dataset...
Discovered 247 images
Training dataset size: 247 images
DataLoader created:
  batch_size: 8
  num_batches: 31
  pin_memory: True


## 6. Initialize Model

In [5]:
print("Initializing model...")

model = HRNet(config['network'])
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Model device: {device}")

Initializing model...
Model parameters: 594,192
Model device: cuda


## 7. Setup Optimizer and Loss

In [6]:
print("Setting up training...")

criterion = torch.nn.MSELoss()

# ---------------------------------------------------------------------------
# CBAM fine-tuning controls (low-complexity, high-impact)
# ---------------------------------------------------------------------------
from DeepNetworks.cbam import CBAM

# Make this cell self-contained (do not depend on later cells).
network_cfg = config.get("network", {})
USE_CBAM = bool(network_cfg.get("use_cbam", False))

USE_CBAM_WARMUP = bool(config['training'].get('use_cbam_warmup', True)) and USE_CBAM
CBAM_WARMUP_EPOCHS = int(config['training'].get('cbam_warmup_epochs', 10))
BASE_LR_JOINT = float(config['training'].get('base_lr_joint', 5e-6))
CBAM_LR_WARMUP = float(config['training'].get('cbam_lr_warmup', 3e-4))
CBAM_LR_JOINT = float(config['training'].get('cbam_lr_joint', 1e-4))


def _split_cbam_and_base_params(model_instance):
    cbam_param_ids = set()
    for module in model_instance.modules():
        if isinstance(module, CBAM):
            for p in module.parameters(recurse=True):
                cbam_param_ids.add(id(p))

    cbam_params_local = []
    base_params_local = []
    for p in model_instance.parameters():
        if id(p) in cbam_param_ids:
            cbam_params_local.append(p)
        else:
            base_params_local.append(p)
    return cbam_params_local, base_params_local


cbam_params, base_params = _split_cbam_and_base_params(model)
cbam_warmup_done = False

print(f"Detected parameter groups:")
print(f"  Base params: {sum(p.numel() for p in base_params):,}")
print(f"  CBAM params: {sum(p.numel() for p in cbam_params):,}")
print(f"  use_cbam: {USE_CBAM}")

if USE_CBAM_WARMUP and len(cbam_params) > 0:
    for p in base_params:
        p.requires_grad = False

    optimizer = optim.Adam(
        cbam_params,
        lr=CBAM_LR_WARMUP,
        betas=(0.9, 0.999)
    )

    print("Optimizer phase: CBAM warmup (base frozen)")
    print(f"  warmup_epochs: {CBAM_WARMUP_EPOCHS}")
    print(f"  cbam_lr_warmup: {CBAM_LR_WARMUP}")
else:
    # Fallback: no warmup or no CBAM -> standard joint optimization.
    if len(cbam_params) > 0:
        optimizer = optim.Adam(
            [
                {'params': base_params, 'lr': BASE_LR_JOINT},
                {'params': cbam_params, 'lr': CBAM_LR_JOINT},
            ],
            betas=(0.9, 0.999)
        )
        print("Optimizer phase: joint fine-tuning (differential LR)")
        print(f"  base_lr_joint: {BASE_LR_JOINT}")
        print(f"  cbam_lr_joint: {CBAM_LR_JOINT}")
    else:
        optimizer = optim.Adam(
            model.parameters(),
            lr=config['training']['lr'],
            betas=(0.9, 0.999)
        )
        print("Optimizer phase: standard (no CBAM params found)")
        print(f"  learning_rate: {config['training']['lr']}")

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['training']['lr_step'],
    gamma=config['training']['lr_decay']
)

print(f"LR decay: {config['training']['lr_decay']} every {config['training']['lr_step']} epochs")
print(f"Loss function: MSELoss")



Setting up training...
Detected parameter groups:
  Base params: 591,818
  CBAM params: 2,374
  use_cbam: True
Optimizer phase: joint fine-tuning (differential LR)
  base_lr_joint: 3e-06
  cbam_lr_joint: 0.0003
LR decay: 0.9 every 5 epochs
Loss function: MSELoss


## 8. Training Loop

In [7]:
# Training run setup (selection is managed by notebooks/weights_manager.ipynb)
import json
from datetime import datetime
from pathlib import Path

WEIGHTS_ROOT = Path("../models/weights")
RUNS_TMP_DIR = WEIGHTS_ROOT / "_runs_tmp"
ACTIVE_SELECTION_PATH = WEIGHTS_ROOT / "active_weight_selection.json"
LAST_TRAINING_CANDIDATE_PATH = WEIGHTS_ROOT / "last_training_candidate.json"

for folder in [WEIGHTS_ROOT, RUNS_TMP_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


def _read_active_selection():
    if not ACTIVE_SELECTION_PATH.exists():
        return None
    try:
        with open(ACTIVE_SELECTION_PATH, "r") as f:
            data = json.load(f)
        path_value = data.get("path")
        if not path_value:
            return None
        candidate = Path(path_value)
        if candidate.exists():
            return candidate
    except Exception:
        return None
    return None


# ---------------------------------------------------------------------------
# USER CONTROLS (edit these only)
# ---------------------------------------------------------------------------
RUN_TAG = ""
ALLOW_SCRATCH_IF_NO_SELECTION = True  # If False, training stops unless weights_manager set an active weight

# Architecture family is inferred from config switch for consistency.
network_cfg = config.get("network", {})
USE_CBAM = bool(network_cfg.get("use_cbam", False))
ACTIVE_ARCH_FAMILY = "CBAM" if USE_CBAM else "Base"

selected_source_weight_path = _read_active_selection()
selected_source_weight_name = selected_source_weight_path.stem if selected_source_weight_path else None

if selected_source_weight_path is not None:
    print(f"Selected source weight from weights manager: {selected_source_weight_path}")
else:
    if not ALLOW_SCRATCH_IF_NO_SELECTION:
        raise RuntimeError(
            "No active weight selection found. Set one in notebooks/weights_manager.ipynb or enable scratch mode."
        )
    print("No active weight selected -> training from scratch")
    

run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_init = "finetune" if selected_source_weight_path else "scratch"
run_name = f"{run_stamp}_{ACTIVE_ARCH_FAMILY.lower()}_{run_init}_{RUN_TAG}"
run_dir = RUNS_TMP_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=True)

managed_best_weights_path = run_dir / "best.pth"
managed_checkpoint_dir = run_dir / "checkpoints"
managed_checkpoint_dir.mkdir(parents=True, exist_ok=True)
managed_checkpoint_meta_path = run_dir / "training_metadata.json"

print("\nRun setup:")
print(f"  run_name: {run_name}")
print(f"  run_dir: {run_dir}")
print(f"  architecture: {ACTIVE_ARCH_FAMILY} (use_cbam={USE_CBAM})")
print(f"  mode: {run_init}")
print("  save/discard: handled in notebooks/weights_manager.ipynb")


Selected source weight from weights manager: D:\GUC\HighResNet model\HighRes-net\models\weights\CBAM\20260423_235944_cbam_CBAM 1.5 bias.pth

Run setup:
  run_name: 20260424_232024_cbam_finetune_
  run_dir: ..\models\weights\_runs_tmp\20260424_232024_cbam_finetune_
  architecture: CBAM (use_cbam=True)
  mode: finetune
  save/discard: handled in notebooks/weights_manager.ipynb


In [8]:
from tqdm import tqdm
import json as json_module
import sys
import shutil
from datetime import datetime

# ============================================================================
# Range + smoothness regularizers
# ============================================================================
def compute_range_loss(sr_output, hr_target):
    """
    Prevent output dynamic-range collapse by matching SR/HR range statistics.
    """
    sr_min = sr_output.min()
    sr_max = sr_output.max()
    sr_range = sr_max - sr_min

    hr_min = hr_target.min()
    hr_max = hr_target.max()

    target_range = 0.6
    range_loss = (sr_range - target_range) ** 2
    min_align = (sr_min - hr_min) ** 2
    max_align = (sr_max - hr_max) ** 2
    return range_loss * 0.5 + (min_align + max_align) * 0.25


def tv_loss(output):
    """Standard TV: penalizes all local variation (can oversmooth edges)."""
    diff_h = torch.abs(output[:, :, 1:, :] - output[:, :, :-1, :])
    diff_w = torch.abs(output[:, :, :, 1:] - output[:, :, :, :-1])
    return torch.mean(diff_h) + torch.mean(diff_w)


def edge_aware_tv_loss(output, hr_target, edge_k=10.0):
    """
    Penalize SR variation mostly where HR is smooth, and relax penalty at true HR edges.
    """
    hr_grad_h = torch.abs(hr_target[:, :, 1:, :] - hr_target[:, :, :-1, :])
    hr_grad_w = torch.abs(hr_target[:, :, :, 1:] - hr_target[:, :, :, :-1])

    edge_weight_h = torch.exp(-edge_k * hr_grad_h)
    edge_weight_w = torch.exp(-edge_k * hr_grad_w)

    sr_grad_h = torch.abs(output[:, :, 1:, :] - output[:, :, :-1, :])
    sr_grad_w = torch.abs(output[:, :, :, 1:] - output[:, :, :, :-1])

    return torch.mean(edge_weight_h * sr_grad_h) + torch.mean(edge_weight_w * sr_grad_w)


def _radial_frequency_mask(h, w, low_ratio, high_ratio, device, dtype):
    """Build a radial band-pass mask in normalized FFT frequency radius [0, 1]."""
    fy = torch.fft.fftfreq(h, d=1.0, device=device).view(h, 1)
    fx = torch.fft.fftfreq(w, d=1.0, device=device).view(1, w)
    r = torch.sqrt(fy * fy + fx * fx)
    r = r / r.max().clamp_min(1e-8)
    mask = ((r >= low_ratio) & (r <= high_ratio)).to(dtype)
    return mask


def freq_band_loss(sr_output, hr_target, low_ratio=0.08, high_ratio=0.35, use_log_magnitude=True):
    """
    Penalize spectral error energy in a selected mid-frequency band.
    This is more targeted than spatial TV for ringing suppression.
    """
    err = sr_output - hr_target
    err_fft = torch.fft.fft2(err, dim=(-2, -1))
    err_mag = torch.abs(err_fft)

    if use_log_magnitude:
        err_mag = torch.log1p(err_mag)

    h = err_mag.shape[-2]
    w = err_mag.shape[-1]
    band = _radial_frequency_mask(h, w, low_ratio, high_ratio, err_mag.device, err_mag.dtype)
    band = band.unsqueeze(0).unsqueeze(0)

    denom = band.sum() * err_mag.shape[0] * err_mag.shape[1]
    return ((err_mag * band) ** 2).sum() / denom.clamp_min(1.0)


# ---------------------------------------------------------------------------
# Lightweight CBAM diagnostics (checks if gates are near-uniform)
# ---------------------------------------------------------------------------
cbam_diag = {
    "channel_means": [],
    "channel_stds": [],
    "spatial_means": [],
    "spatial_stds": [],
}
cbam_diag_hooks = []


def _cbam_gate_hook(kind):
    def _hook(module, inputs, output):
        out = output.detach()
        if kind == "channel":
            cbam_diag["channel_means"].append(float(out.mean().item()))
            cbam_diag["channel_stds"].append(float(out.std().item()))
        else:
            cbam_diag["spatial_means"].append(float(out.mean().item()))
            cbam_diag["spatial_stds"].append(float(out.std().item()))
    return _hook


if USE_CBAM:
    for name, module in model.named_modules():
        if name.endswith("channel_attn.gate"):
            cbam_diag_hooks.append(module.register_forward_hook(_cbam_gate_hook("channel")))
        elif name.endswith("spatial_attn.gate"):
            cbam_diag_hooks.append(module.register_forward_hook(_cbam_gate_hook("spatial")))
    print(f"CBAM diagnostics: registered {len(cbam_diag_hooks)} gate hooks")

print("\nStarting training with configurable composite loss...")
print("=" * 70)
print("Loss = MSE + lambda_range * RangeLoss + lambda_smooth * SmoothnessLoss")
print("SmoothnessLoss mode: none | tv | edge_aware_tv | freq_band")
print("=" * 70)

num_epochs = config['training']['num_epochs']
lambda_range = config['training'].get('lambda_range', 0.003)

smoothness_mode = config['training'].get('smoothness_mode', 'tv').lower()
lambda_tv = config['training'].get('lambda_tv', 0.0)
lambda_edge_tv = config['training'].get('lambda_edge_tv', 0.0)
lambda_freq = config['training'].get('lambda_freq', 0.0)
edge_aware_k = float(config['training'].get('edge_aware_k', 10.0))
freq_low_ratio = float(config['training'].get('freq_low_ratio', 0.08))
freq_high_ratio = float(config['training'].get('freq_high_ratio', 0.35))
freq_use_log_magnitude = bool(config['training'].get('freq_use_log_magnitude', True))

if smoothness_mode == 'edge_aware_tv':
    lambda_smooth = lambda_edge_tv if lambda_edge_tv > 0 else lambda_tv
elif smoothness_mode == 'tv':
    lambda_smooth = lambda_tv
elif smoothness_mode == 'freq_band':
    lambda_smooth = lambda_freq
else:
    lambda_smooth = 0.0

best_loss = float('inf')
best_weights_path = managed_best_weights_path
checkpoint_meta_path = managed_checkpoint_meta_path
checkpoint_dir = managed_checkpoint_dir

# FIXED — handles decoder transition safely
source_state_dict = torch.load(
    selected_source_weight_path, map_location=device, weights_only=False
)

model_state = model.state_dict()
filtered_state_dict = {}
skipped_keys = []

for key, ckpt_tensor in source_state_dict.items():
    if key in model_state:
        if ckpt_tensor.shape == model_state[key].shape:
            filtered_state_dict[key] = ckpt_tensor
        else:
            skipped_keys.append(f"{key}: checkpoint {tuple(ckpt_tensor.shape)} → model {tuple(model_state[key].shape)}")
    else:
        skipped_keys.append(f"{key}: not in current model")

missing_keys, unexpected_keys = model.load_state_dict(filtered_state_dict, strict=False)

# --- this goes at the END of your load cell, after load_state_dict ---

test_param = model.encode.init_layer[0].weight
print(f"\nLoad verification:")
print(f"  File: {selected_source_weight_path}")
print(f"  Size: {selected_source_weight_path.stat().st_size / 1e6:.2f} MB")
print(f"  Encoder init_layer mean: {test_param.mean():.6f}")
print(f"  Encoder init_layer std:  {test_param.std():.6f}")
print(f"  Parameter magnitude:     {sum(p.abs().mean().item() for p in model.parameters()):.4f}")
print(f"  Keys loaded: {len(filtered_state_dict)}")

# Interpret the result automatically
enc_std = test_param.std().item()
if enc_std < 0.01:
    print("  STATUS: ⚠️  Load likely failed — weights look random (std too low)")
elif enc_std > 0.02:
    print("  STATUS: ✓  Load succeeded — weights look trained")
else:
    print("  STATUS: ⚠️  Uncertain — check initial epoch 1 MSE")

print(f"Source load complete.")
print(f"  Keys loaded:  {len(filtered_state_dict)}")
print(f"  Keys skipped: {len(skipped_keys)}")
for s in skipped_keys:
    print(f"    {s}")

# Verify the load actually worked
first_mse_check = sum(p.abs().mean().item() for p in model.parameters())
print(f"Parameter magnitude check: {first_mse_check:.4f}  (should be >> 0)")

# Build phase optimizer/scheduler *after* source weight load so phase setup is based
# on the actual loaded model state and no pre-load optimizer state is reused.
cbam_params, base_params = _split_cbam_and_base_params(model)
cbam_warmup_done = False

if USE_CBAM_WARMUP and len(cbam_params) > 0 and CBAM_WARMUP_EPOCHS > 0:
    for p in base_params:
        p.requires_grad = False
    for p in cbam_params:
        p.requires_grad = True

    optimizer = optim.Adam(
        cbam_params,
        lr=CBAM_LR_WARMUP,
        betas=(0.9, 0.999)
    )
else:
    for p in model.parameters():
        p.requires_grad = True

    if len(cbam_params) > 0:
        optimizer = optim.Adam(
            [
                {'params': base_params, 'lr': BASE_LR_JOINT},
                {'params': cbam_params, 'lr': CBAM_LR_JOINT},
            ],
            betas=(0.9, 0.999)
        )
        cbam_warmup_done = True
    else:
        optimizer = optim.Adam(
            model.parameters(),
            lr=config['training']['lr'],
            betas=(0.9, 0.999)
        )
        cbam_warmup_done = True

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=config['training']['lr_step'],
    gamma=config['training']['lr_decay']
)

print(f"\nlambda_range:            {lambda_range}")
print(f"smoothness_mode:         {smoothness_mode}")
print(f"lambda_smooth:           {lambda_smooth}")
print(f"edge_aware_k:            {edge_aware_k}")
print(f"freq_low/high ratios:    {freq_low_ratio} / {freq_high_ratio}")
print(f"freq_use_log_magnitude:  {freq_use_log_magnitude}")
print(f"best_weights_path:       {best_weights_path}")
print(f"checkpoint_dir:          {checkpoint_dir}")
print(f"cbam_warmup_enabled:     {USE_CBAM_WARMUP}")
print(f"cbam_warmup_epochs:      {CBAM_WARMUP_EPOCHS}")
print(f"base_lr_joint:           {BASE_LR_JOINT}")
print(f"cbam_lr_warmup/joint:    {CBAM_LR_WARMUP} / {CBAM_LR_JOINT}\n")

train_losses = []
train_mse_losses = []
train_range_losses = []
train_tv_losses = []
train_edge_tv_losses = []
train_freq_losses = []
epoch_times = []

pbar = tqdm(
    range(1, num_epochs + 1),
    desc="Training",
    unit="epoch",
    file=sys.stdout,
    ncols=100,
    disable=False,
    bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
)

for epoch in pbar:
    epoch_start = time.time()

    # Transition from CBAM-only warmup to joint fine-tuning.
    if USE_CBAM_WARMUP and (not cbam_warmup_done) and (epoch == CBAM_WARMUP_EPOCHS + 1) and (len(cbam_params) > 0):
        completed_epochs = epoch - 1

        for p in model.parameters():
            p.requires_grad = True

        cbam_params, base_params = _split_cbam_and_base_params(model)
        optimizer = optim.Adam(
            [
                {'params': base_params, 'lr': BASE_LR_JOINT},
                {'params': cbam_params, 'lr': CBAM_LR_JOINT},
            ],
            betas=(0.9, 0.999)
        )

        # Keep LR schedule continuous across warmup -> joint transition.
        for param_group in optimizer.param_groups:
            param_group.setdefault('initial_lr', param_group['lr'])

        scheduler = optim.lr_scheduler.StepLR(
            optimizer,
            step_size=config['training']['lr_step'],
            gamma=config['training']['lr_decay'],
            last_epoch=completed_epochs - 1
        )

        cbam_warmup_done = True
        pbar.write("Switched to joint fine-tuning (base unfrozen + differential LR, scheduler preserved)")

    model.train()
    batch_losses = []
    batch_mse_losses = []
    batch_range_losses = []
    batch_tv_losses = []
    batch_edge_tv_losses = []
    batch_freq_losses = []

    cbam_diag["channel_means"].clear()
    cbam_diag["channel_stds"].clear()
    cbam_diag["spatial_means"].clear()
    cbam_diag["spatial_stds"].clear()

    for batch_idx, (lrs, alphas, hrs, hr_maps, names) in enumerate(train_loader):
        lrs = lrs.float().to(device)
        alphas = alphas.float().to(device)
        hrs = hrs.float().to(device)

        sr_output = model(lrs, alphas)

        if epoch == 1 and batch_idx == 0:
            print("\nShape check:")
            print(f"  LR input shape: {lrs.shape}")
            print(f"  SR output shape: {sr_output.shape}")
            print(f"  HR target shape: {hrs.shape}")

        mse_loss = criterion(sr_output, hrs)
        range_loss = compute_range_loss(sr_output, hrs)
        tv_reg = tv_loss(sr_output)
        edge_tv_reg = edge_aware_tv_loss(sr_output, hrs, edge_k=edge_aware_k)
        freq_reg = freq_band_loss(
            sr_output,
            hrs,
            low_ratio=freq_low_ratio,
            high_ratio=freq_high_ratio,
            use_log_magnitude=freq_use_log_magnitude,
        )

        if smoothness_mode == 'edge_aware_tv':
            smooth_loss = edge_tv_reg
        elif smoothness_mode == 'tv':
            smooth_loss = tv_reg
        elif smoothness_mode == 'freq_band':
            smooth_loss = freq_reg
        else:
            smooth_loss = torch.zeros_like(mse_loss)

        loss = mse_loss + lambda_range * range_loss + lambda_smooth * smooth_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.item())
        batch_mse_losses.append(mse_loss.item())
        batch_range_losses.append(range_loss.item())
        batch_tv_losses.append(tv_reg.item())
        batch_edge_tv_losses.append(edge_tv_reg.item())
        batch_freq_losses.append(freq_reg.item())

    epoch_loss = np.mean(batch_losses)
    epoch_mse_loss = np.mean(batch_mse_losses)
    epoch_range_loss = np.mean(batch_range_losses)
    epoch_tv_loss = np.mean(batch_tv_losses)
    epoch_edge_tv_loss = np.mean(batch_edge_tv_losses)
    epoch_freq_loss = np.mean(batch_freq_losses)

    train_losses.append(epoch_loss)
    train_mse_losses.append(epoch_mse_loss)
    train_range_losses.append(epoch_range_loss)
    train_tv_losses.append(epoch_tv_loss)
    train_edge_tv_losses.append(epoch_edge_tv_loss)
    train_freq_losses.append(epoch_freq_loss)

    scheduler.step()

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)

    postfix = {
        'loss': f'{epoch_loss:.6f}',
        'mse': f'{epoch_mse_loss:.6f}',
        'range': f'{epoch_range_loss:.4f}',
        'tv': f'{epoch_tv_loss:.4f}',
        'e_tv': f'{epoch_edge_tv_loss:.4f}',
        'freq': f'{epoch_freq_loss:.4f}',
        'time': f'{epoch_time:.1f}s'
    }

    if cbam_diag["spatial_stds"]:
        postfix['cbam_sstd'] = f"{np.mean(cbam_diag['spatial_stds']):.4f}"

    pbar.set_postfix(postfix)

    if cbam_diag["spatial_means"] and (epoch == 1 or epoch % 5 == 0 or epoch == num_epochs):
        channel_mean = np.mean(cbam_diag["channel_means"]) if cbam_diag["channel_means"] else float('nan')
        channel_std = np.mean(cbam_diag["channel_stds"]) if cbam_diag["channel_stds"] else float('nan')
        spatial_mean = np.mean(cbam_diag["spatial_means"]) if cbam_diag["spatial_means"] else float('nan')
        spatial_std = np.mean(cbam_diag["spatial_stds"]) if cbam_diag["spatial_stds"] else float('nan')
        pbar.write(
            f"CBAM gate stats @ epoch {epoch} -> "
            f"channel(mean/std): {channel_mean:.4f}/{channel_std:.4f}, "
            f"spatial(mean/std): {spatial_mean:.4f}/{spatial_std:.4f}"
        )

    sys.stdout.flush()

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), best_weights_path)
        pbar.write(f"  Best loss updated at epoch {epoch}")

    if epoch % 25 == 0:
        checkpoint_path = checkpoint_dir / f"checkpoint_epoch_{epoch}.pth"
        torch.save(model.state_dict(), checkpoint_path)
        pbar.write(f"  Checkpoint saved: epoch_{epoch}.pth")

    if epoch % 10 == 0:
        meta = {
            'run_name': run_name,
            'source_weight': str(selected_source_weight_path) if selected_source_weight_path else None,
            'last_saved_epoch': epoch,
            'best_loss': float(best_loss),
            'current_loss': float(epoch_loss),
            'current_mse': float(epoch_mse_loss),
            'current_range': float(epoch_range_loss),
            'current_tv': float(epoch_tv_loss),
            'current_edge_tv': float(epoch_edge_tv_loss),
            'current_freq': float(epoch_freq_loss),
            'smoothness_mode': smoothness_mode,
            'lambda_smooth': float(lambda_smooth),
            'edge_aware_k': float(edge_aware_k),
            'freq_low_ratio': float(freq_low_ratio),
            'freq_high_ratio': float(freq_high_ratio),
            'freq_use_log_magnitude': bool(freq_use_log_magnitude),
            'num_epochs': num_epochs,
            'dataset_size': len(train_dataset),
            'timestamp': time.time()
        }
        with open(checkpoint_meta_path, 'w') as f:
            json_module.dump(meta, f, indent=2)

pbar.close()

for hook in cbam_diag_hooks:
    hook.remove()

print("\n" + "=" * 70)
print("Training complete")
print(f"Best combined loss: {best_loss:.6f}")
print(f"Run best weights: {best_weights_path}")
print(f"Run metadata: {checkpoint_meta_path}")
print(f"Run checkpoints: {checkpoint_dir}")
print("\nTraining Statistics:")
print(f"  Total time: {sum(epoch_times) / 60:.1f} minutes")
print(f"  Avg time per epoch: {np.mean(epoch_times):.1f}s")
print("\nMSE Loss (pixel accuracy):")
print(f"  Initial: {train_mse_losses[0]:.6f}")
print(f"  Final:   {train_mse_losses[-1]:.6f}")
print(f"  Improvement: {(train_mse_losses[0] - train_mse_losses[-1]) / train_mse_losses[0] * 100:.1f}%")
print("\nRange Loss (dynamic range):")
print(f"  Initial: {train_range_losses[0]:.6f}")
print(f"  Final:   {train_range_losses[-1]:.6f}")
print(f"  Improvement: {(train_range_losses[0] - train_range_losses[-1]) / train_range_losses[0] * 100:.1f}%")
print("\nSmoothness diagnostics:")
print(f"  TV initial/final:       {train_tv_losses[0]:.6f} -> {train_tv_losses[-1]:.6f}")
print(f"  Edge-TV initial/final:  {train_edge_tv_losses[0]:.6f} -> {train_edge_tv_losses[-1]:.6f}")
print(f"  Freq initial/final:     {train_freq_losses[0]:.6f} -> {train_freq_losses[-1]:.6f}")
print(f"  Active smoothness mode: {smoothness_mode} (lambda={lambda_smooth})")
print("=" * 70)

# Keep/discard is intentionally deferred to weights_manager.ipynb.
last_trained_weight_path = best_weights_path
last_saved_weight_path = None
post_training_promotion_done = False

candidate_summary = {
    "run_name": run_name,
    "run_dir": str(run_dir),
    "best_weight_path": str(last_trained_weight_path),
    "checkpoint_dir": str(checkpoint_dir),
    "meta_path": str(checkpoint_meta_path),
    "source_weight": str(selected_source_weight_path) if selected_source_weight_path else None,
    "arch_family": ACTIVE_ARCH_FAMILY,
    "created_at": datetime.now().isoformat(),
    "best_training_loss": float(best_loss)
}
with open(LAST_TRAINING_CANDIDATE_PATH, "w") as f:
    json.dump(candidate_summary, f, indent=2)

print("\nTraining finished. Review plots/diagnostics, then save/discard via notebooks/weights_manager.ipynb")
print(f"last_trained_weight_path: {last_trained_weight_path}")
print(f"candidate summary file:   {LAST_TRAINING_CANDIDATE_PATH}")

CBAM diagnostics: registered 4 gate hooks

Starting training with configurable composite loss...
Loss = MSE + lambda_range * RangeLoss + lambda_smooth * SmoothnessLoss
SmoothnessLoss mode: none | tv | edge_aware_tv | freq_band

Load verification:
  File: D:\GUC\HighResNet model\HighRes-net\models\weights\CBAM\20260423_235944_cbam_CBAM 1.5 bias.pth
  Size: 2.39 MB
  Encoder init_layer mean: 0.001688
  Encoder init_layer std:  0.133543
  Parameter magnitude:     9.0418
  Keys loaded: 41
  STATUS: ✓  Load succeeded — weights look trained
Source load complete.
  Keys loaded:  41
  Keys skipped: 0
Parameter magnitude check: 9.0418  (should be >> 0)

lambda_range:            0.003
smoothness_mode:         freq_band
lambda_smooth:           0.01
edge_aware_k:            10.0
freq_low/high ratios:    0.12 / 0.28
freq_use_log_magnitude:  True
best_weights_path:       ..\models\weights\_runs_tmp\20260424_232024_cbam_finetune_\best.pth
checkpoint_dir:          ..\models\weights\_runs_tmp\20260424

KeyboardInterrupt: 

## 9. Plot Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training loss
axes[0].plot(train_losses, linewidth=2, color='blue')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Epoch time
axes[1].plot(epoch_times, linewidth=2, color='orange')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Time (seconds)', fontsize=12)
axes[1].set_title('Epoch Computation Time', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTraining Statistics:")
print(f"  Total time: {sum(epoch_times)/60:.1f} minutes")
print(f"  Avg time per epoch: {np.mean(epoch_times):.1f}s")
print(f"  Loss decreased by: {(train_losses[0] - train_losses[-1]) / train_losses[0] * 100:.1f}%")